# 12. Grover's Algorithm: The Quantum Search Speedup

In this lab you will learn how to build and use Grover circuit generating functions.

If you have an unsorted database of $N$ items and want to find a specific marked item, a classical computer must check each item one by one. On average, this takes $N/2$ steps, and in the worst case, $N$ steps. This is an $O(N)$ time complexity.

Grover's Algorithm provides a **quadratic speedup**, finding the marked item in roughly $\sqrt{N}$ steps. For a database of 1 million items, a classical computer might take 500,000 attempts, while a quantum computer using Grover's Algorithm would take only about 1,000.

## 12.1. The Geometric Intuition
Grover's Algorithm works by iteratively rotating the quantum state vector closer and closer to the winning state. We can visualize this in a 2D plane defined by two orthogonal vectors:
1. $|w\rangle$: The winning (marked) state(s) we are looking for.
2. $|s'\rangle$: A uniform superposition of all the *wrong* states.

Our starting state, $|s\rangle$, is a uniform superposition of *all* possible states (created by applying Hadamard gates to all qubits). This initial state $|s\rangle$ is very close to $|s'\rangle$ and almost perpendicular to $|w\rangle$.

[Image of Grover's algorithm geometric rotation in state space]

The algorithm consists of repeating two operators to rotate $|s\rangle$ towards $|w\rangle$:

### 1. The Phase Oracle ($U_w$)
The Oracle is a "black box" function that recognizes the winning state. When the Oracle sees the winning state $|w\rangle$, it flips its phase (multiplying its amplitude by -1). For all other states, it does nothing.
Geometrically, this is a **reflection about the $|s'\rangle$ axis**.

**How do we build this?** We use the concept of **Phase Kickback**. By applying a controlled-operation where the target qubit is in the $|-\rangle$ state, the eigenvalue of the operation (a negative sign for the marked state) is "kicked back" into the phase of the control qubits without altering their measurement probabilities.

[Image of Phase Kickback quantum circuit mechanism]

### 2. The Diffuser ($U_s$)
Flipping the phase of the winning state doesn't change its measurement probability (since probability is the magnitude squared). We need a second step to turn this phase difference into an amplitude difference.

The Diffuser performs an **inversion about the mean** (or a reflection about the average state $|s\rangle$). Because the Oracle made the amplitude of the winning state negative, the new average amplitude is slightly lower. Reflecting the negative winning state around this new, lower average aggressively amplifies its magnitude, while shrinking the magnitudes of all the wrong states.

[Image of Grover's algorithm Diffuser reflection]

By applying the Oracle and the Diffuser iteratively $\approx \frac{\pi}{4}\sqrt{\frac{N}{M}}$ times (where $M$ is the number of marked items), the state vector perfectly aligns with the winning state $|w\rangle$, guaranteeing a successful measurement!

## 12.2. Implementation of Grover's algorithm in Qiskit

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# Importing standard Qiskit libraries
from qiskit import QuantumCircuit, transpile
from qiskit_aer import Aer
from qiskit.visualization import *
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
from qiskit.quantum_info import Statevector

# Import custom visualization tools
try:
    from qc_interactive_education_package import DimensionalCircleNotation
    HAS_CUSTOM_TOOLS = True
except ImportError:
    HAS_CUSTOM_TOOLS = False
    print("Custom visualization package not found. DCN visualizations will be skipped.")

### Step 1: Building the Phase Oracle

<div style="background-color: #e31b4c; color: #ffffff; padding: 15px; border-radius: 5px; border: 1px solid #042c58;">
<strong>Your Task: Finish creating the "phase_oracle" function that does the following:</strong><br>
<ul>
    <li>Generates the oracle matrix to flip the phase of the marked elements.</li>
    <li>Transforms the matrix to the corresponding unitary gate.</li>
</ul>
In practicality, the phase of the winning state(s) get(s) flipped.
</div>

In [ ]:
def phase_oracle(n, indices_to_mark, name='Oracle'):

    # Initialize a quantum circuit with n qubits and the specified name.
    qc = QuantumCircuit(n, name=name)

    # Get an identity matrix of 2^n dimension.
    oracle_matrix = ... # YOUR CODE HERE

    # For each key in the list of keys, flip the sign of the key.
    for index_to_mark in indices_to_mark:
        # YOUR CODE HERE
        pass

    # Apply the matrix to the circuit as a unitary gate with range of qubits being [0, ..., n].
    # YOUR CODE HERE

    return qc

### Step 2: Building the Diffuser

<div style="background-color: #e31b4c; color: #ffffff; padding: 15px; border-radius: 5px; border: 1px solid #042c58;">
<strong>Your Task: Finish creating the "diffuser" function that does the following:</strong><br>
<ul>
    <li>Prepares the diffuser circuit and generates the matrix to flip the state 0.</li>
    <li>Physically, the diffuser amplifies the amplitudes of the winning states.</li>
</ul>
</div>

In [ ]:
def diffuser(n, name='Diffuser'):
    # defining a list of [0, ..., (2**n) -1]
    all_num = list(range(2 ** n))
    # defining a list of [1, ..., (2**n) -1]
    except_zero = all_num[1:]

    # Initialize a quantum circuit with n qubits and the specified name.
    qc = QuantumCircuit(n, name=name)

    # Prepare the state |s> = H|0>.
    # YOUR CODE HERE

    # Generate the diffuser matrix using the phase_oracle function.
    diffuser_matrix = phase_oracle(n, except_zero)

    # Apply the diffuser matrix => D|s>.
    # YOUR CODE HERE

    # Apply the H gate to the circuit => HD|s>.
    # YOUR CODE HERE

    return qc

### Step 3: Constructing the Full Circuit

<div style="background-color: #e31b4c; color: #ffffff; padding: 15px; border-radius: 5px; border: 1px solid #042c58;">
<strong>Your Task: Finish creating the "grover_search_circ" function that does the following:</strong><br>
<ul>
    <li>Prepares the entire circuit for the Grover's search algorithm.</li>
    <li>Searches for the marked elements in the range (0, ..., 2^(n-1)).</li>
</ul>
</div>

In [ ]:
def grover_search_circ(n, marked):
    # Initialize a quantum circuit with n qubits and classical bits.
    qc = QuantumCircuit(n, n)

    # let M be the dimension of the list of elements.
    M = len(marked)

    # Let N be the total number of elements that can be represented with n qubits.
    N = 2 ** n

    # Get the angle of rotation.
    theta = ... # YOUR CODE HERE

    # Get the number of rounds needed.
    rounds = ... # YOUR CODE HERE

    list_of_state_vectors = []

    # Print the information:
    print(f"Number of qubits: {n}")
    print(f"Key(s) to search: {marked}")
    print(f"Number of rounds needed: {rounds}\n")

    # Step 1: Prepare the superposition states of each qubit.
    # YOUR CODE HERE

    # For all the rounds...
    for _ in range(rounds):
        # Step 2: Apply the phase oracle.
        # YOUR CODE HERE
        # Don't forget to append the statevector: list_of_state_vectors.append(Statevector.from_instruction(qc))

        # Step 3: Apply the diffuser.
        # YOUR CODE HERE
        # Don't forget to append the statevector: list_of_state_vectors.append(Statevector.from_instruction(qc))

    # Final step: Measure the qubits.
    # YOUR CODE HERE

    return qc, list_of_state_vectors

### Step 4: Execution and Visualization

<div style="background-color: #e31b4c; color: #ffffff; padding: 15px; border-radius: 5px; border: 1px solid #042c58;">
<strong>Your Task: Test the function with various input variables, run the code on a local simulator, and retrieve and print the results.</strong><br>
</div>

In [ ]:
# 1. Ask for user input and generate the circuit
n = int(input("Enter the number of qubits (int>0):"))
marked = list(map(int, input('Enter the elements (int>0) separated by spaces:\n').split()))
qc, list_of_state_vectors = grover_search_circ(n, marked)

# 2. Run it in the simulator
simulator = Aer.get_backend('aer_simulator')
transpiled_circuit = transpile(qc, simulator)

# Define the job using the transpiled circuit with 1024 shots
job = ... # YOUR CODE HERE

# Retrieve the result
result = ... # YOUR CODE HERE

# 3. Retrieve and print the results
counts = ... # YOUR CODE HERE

print(counts)
display(plot_histogram(counts))

## 12.3. Visualizing the algorithm

In [ ]:
from qc_interactive_education_package.quantum_library import QuantumCurriculum
from qc_interactive_education_package import launch_app

# 1. Retrieve the pre-compiled algorithms from the curriculum
algorithms = QuantumCurriculum.get_algorithms()

# 2. Extract the exact 5-qubit Grover matrix
grover_5q = algorithms["Shor's Algorithm: Factor 15 (8Q)"]

# 3. Instantiate the viewer directly into the active DOM
# The viewer will automatically deduce the initial and target states
# based on the architectural updates we implemented.
launch_app(
    mode='algorithm',
    num_qubits=8,
    preloaded_circuit=grover_5q,
    show_circuit=True,
    show_annotations=True,
    show_final_state=False
)

Initializing Quantum Education Suite SPA (Algorithm Mode)...
Starting local server... A browser window will open automatically once ready.


## 12.4. Conclusion and Outlook

In this lab, we successfully constructed and executed Grover's Algorithm, observing how amplitude amplification provides a quadratic speedup over classical brute-force search methods.

While searching a simple list is a great way to understand the mechanism, the true power of Grover's Algorithm lies in evaluating complex **Boolean functions**. Many real-world problems in cryptography, optimization, and logistics fall into a category of computer science known as NP (Nondeterministic Polynomial time). These are problems where evaluating a potential solution to see if it is correct is computationally easy, but finding that correct solution among millions of possibilities is extraordinarily hard.

The Phase Oracle is essentially a quantum implementation of such a Boolean function, $f(x)$.
* If $x$ is not the solution, $f(x) = 0$, and the Oracle does nothing.
* If $x$ is the correct solution, $f(x) = 1$, and the Oracle applies a phase flip.

By replacing our simple matrix-based Oracle with a logical circuit that evaluates a complex mathematical condition (like checking if a password hash matches or solving a Satisfiability/SAT problem), we can use the exact same Diffuser we built today to amplify the correct answer. This turns Grover's Algorithm from a theoretical database search into a universal tool for reversing complex functions and solving computationally dense puzzles!